# Exercício Prático — Enclose.horse (Busca)

**Nome:** Pedro Henrique de Oliveira Duarte

## Introdução

Neste notebook, é implementado um ambiente do jogo enclose.horse a partir de arquivos .txt e algoritmos de busca para tentar obter o caminho mínimo caso o cavalo consiga fugir e, caso contrário, obter a pontuação obtida.

## 2) Definição do problema de busca a ser resolvido

- Estado inicial: Posição (i, j) rotulada por C;
- Conjunto de ações: Andar para cima, andar para esquerda, andar para a direita ou andar para baixo;
- Modelo de transição: dado um estado s = (i , j), temos as seguintes transições:
    - T(s, cima) = (i - 1, j);
    - T(s, baixo) = (i + 1, j);
    - T(s, esquerda) = (i, j - 1);
    - T(s, direita) = (i, j + 1);
    <br>
    tal que cada coordenada destino acima não é rotulada por + ou % e não ultrapassa os limites de altura e largura da matriz. 
- Estados objetivo: qualquer estado (i, j) rotulado por vazio, J, M ou A tal que i = 0 ou i = h - 1 ou j = 0 ou j = w - 1, sendo h a altura da matriz e w a largura. Ou seja, qualquer estado que representa uma posição que não é obstáculo e se localiza nas bordas da matriz;
- Custo dos passos: 1;
- Conjunto de todos os estados: todos os estados (i, j) não rotulados por + ou %. Ou seja, todos os estados que representam posições que não são obstáculo. 

## 3) Implementar o ambiente

Nesta seção, implemento o carregamento do arquivo de estado `.txt` na forma de matriz e as funções compartilhadas pelos algoritmos de busca. Além disso, crio uma classe No que será utilizada para representar os estados nos algoritmos de busca.

Um ponto interessante é que, na função de transição, a ordem em que as possíveis ações são testadas e traduzidas em estados afeta o estado objetivo retornado pelos algoritmos, pois muda a ordem em que os estados são adicionados na fronteira. A otimalidade não é afetada, a BFS e o A* ainda vão encontrar o caminho mínimo, mas poderiam encontrar caminhos e estado objetivo diferente em caso de múltiplos caminhos mínimos dependendo da ordem em que as ações são aplicadas na função de transição.

No cálculo de pontuação, me aproveito da estrutura implementada para BFS para executar esse mesmo algoritmo para calcular os pontos. Enquanto na busca pelo caminho mínimo a busca para quando encontramos um estado final, no calculo de pontuação a busca só para quando a fronteira está vazia. Ou seja, o cavalo percorre todos as posições possíveis e vai somando seus pontos no caminho conforme os rótulos.


In [16]:
from pathlib import Path
from dataclasses import dataclass
from collections import deque


class No:
    def __init__(self, i, j, rotulo):
        self.i = i
        self.j = j
        self.rotulo = rotulo


@dataclass(frozen=True)
class Ambiente:
    largura: int
    altura: int
    matriz: list[list[str]]
    posicao_cavalo: tuple[int, int] | None

    @classmethod
    def from_txt(cls, caminho_arquivo: str | Path) -> "Ambiente":
        caminho = Path(caminho_arquivo)
        linhas = caminho.read_text(encoding="utf-8").splitlines()

        largura, altura = map(int, linhas[0].split())

        linhas_tabuleiro = linhas[1 : 1 + altura]

        matriz: list[list[str]] = []
        posicao_cavalo: tuple[int, int] | None = None
        for i, linha in enumerate(linhas_tabuleiro):
            row = list(linha)
            matriz.append(row)

            if "C" in row:
                j = row.index("C")
                posicao_cavalo = (i, j)

        return cls(
            largura=largura,
            altura=altura,
            matriz=matriz,
            posicao_cavalo=posicao_cavalo,
        )

    def printar_matriz(self):
        for linha in self.matriz:
            print("".join(linha))

    def estado_inicial(self):
        return self.posicao_cavalo

    def estado_final(self, no) -> bool:
        i = no.i
        j = no.j
        rotulo = getattr(no, "rotulo", self.matriz[i][j])
        if rotulo == " " or rotulo == "A" or rotulo == "J" or rotulo == "M":
            if i == 0 or i == self.altura - 1 or j == 0 or j == self.largura - 1:
                return True
        return False

    def funcao_transicao(self, no):
        i, j = no.i, no.j
        estados = []
        obstaculos = {"%", "+"}
        if i + 1 < self.altura and self.matriz[i + 1][j] not in obstaculos:  # baixo
            estados.append(No(i + 1, j, self.matriz[i + 1][j]))
        if i - 1 >= 0 and self.matriz[i - 1][j] not in obstaculos:  # cima
            estados.append(No(i - 1, j, self.matriz[i - 1][j]))
        if j + 1 < self.largura and self.matriz[i][j + 1] not in obstaculos:  # direita
            estados.append(No(i, j + 1, self.matriz[i][j + 1]))
        if j - 1 >= 0 and self.matriz[i][j - 1] not in obstaculos:  # esquerda
            estados.append(No(i, j - 1, self.matriz[i][j - 1]))

        return estados

    def calcular_pontuacao(self):
        i0, j0 = self.posicao_cavalo
        no_inicial = No(i0, j0, self.matriz[i0][j0])

        fronteira = FronteiraBFS()
        fronteira.adicionar(no_inicial)
        visitados = {(i0, j0)}
        pontuacao = 0

        while not fronteira.esta_vazia():
            no = fronteira.remover()
            celula = no.rotulo

            pontuacao += 1
            if celula == "J":
                pontuacao += 3
            elif celula == "M":
                pontuacao += 10
            elif celula == "A":
                pontuacao -= 5

            for no_filho in self.funcao_transicao(no):
                coord = (no_filho.i, no_filho.j)
                if coord not in visitados:
                    visitados.add(coord)
                    fronteira.adicionar(no_filho)

        return pontuacao


def carregar_ambiente_txt(caminho_arquivo):
    return Ambiente.from_txt(caminho_arquivo)

In [17]:
# Teste rápido: carregamento e impressão da matriz
ambiente = carregar_ambiente_txt("estados/entice.txt")
ambiente.printar_matriz()
print("Posição do cavalo:", ambiente.posicao_cavalo)

%%%% % %   %%%%
%%     %     %%
%  J   % %    %
%  %   %      %
  % %  %   %%  
 %   % %  %  % 
               
%%%%%% C %%%%%%
               
 %   % %   %%A 
   %   %  %%%% 
% %%%% %      %
%  M%  % %%%  %
%%     %     %%
%%%%   %   %%%%
Posição do cavalo: (7, 7)


## 4 e 5) Implementar e executar algoritmos de busca (BFS, DFS e A*)

Nesta seção, implemento e excuto os algortimos BFS, DFS e A*. Todos retornam também estatísticas de execução: número de nós alcançados (descobertos) e nós expandidos. Além disso, implemento uma função para reconstruir os caminhos percorridos pelos algoritmos para exibição posterior.

Utilizo classes separadas para implementar as estruturas de dados das fronteiras de cada algoritmo a partir de métodos de adicionar e remover.

No BFS, temos uma fila, lógica de adição e remoção FIFO.
No DFS, temos uma pilha, lógica de adição e remoção LIFO.
No AStar, temos uma fila de prioridade, remoção com menor prioridade.

Implemento as buscas em classes separadas também. Procurei me basear nos algoritmos disponibilizados nos slides das aulas.

No A*, escolhi a distância euclidiana até um estado objetivo como heurística admissível.

In [18]:
import heapq
import math

def reconstruir_caminho(pais, objetivo_coord):
    caminho = []
    atual = objetivo_coord
    while atual is not None:
        caminho.append(atual)
        atual = pais.get(atual)
    caminho.reverse()
    return caminho


class FronteiraBFS:
    def __init__(self):
        self.fronteira = []

    def adicionar(self, no):
        self.fronteira.append(no)

    def remover(self):
        return self.fronteira.pop(0)

    def esta_vazia(self):
        return len(self.fronteira) == 0


class FronteiraDFS:
    def __init__(self):
        self.fronteira = []

    def adicionar(self, no):
        self.fronteira.append(no)

    def remover(self):
        return self.fronteira.pop()

    def esta_vazia(self):
        return len(self.fronteira) == 0


class FronteiraAStar:
    def __init__(self):
        self.fronteira = []
        self._contador = 0

    def adicionar(self, f, h, coord, no):
        self._contador += 1
        heapq.heappush(self.fronteira, (f, h, self._contador, coord, no))

    def remover(self):
        return heapq.heappop(self.fronteira)

    def esta_vazia(self):
        return len(self.fronteira) == 0


class Bfs:
    def bfs(matriz, estado_inicial):
        nos_alcancados = set()
        nos_expandidos = 0
        pais = {}

        estado_inicial = No(
            estado_inicial[0],
            estado_inicial[1],
            matriz[estado_inicial[0]][estado_inicial[1]],
        )
        coord_inicial = (estado_inicial.i, estado_inicial.j)
        pais[coord_inicial] = None

        if ambiente.estado_final(estado_inicial):
            return estado_inicial, [coord_inicial], 1, 0

        fronteira = FronteiraBFS()
        fronteira.adicionar(estado_inicial)
        nos_alcancados.add(coord_inicial)
        nos_alcancados_qtd = 1

        while not fronteira.esta_vazia():
            no = fronteira.remover()
            nos_expandidos += 1

            for no_filho in ambiente.funcao_transicao(no):
                coord = (no_filho.i, no_filho.j)
                if coord not in nos_alcancados:
                    pais[coord] = (no.i, no.j)
                    if ambiente.estado_final(no_filho):
                        caminho = reconstruir_caminho(pais, coord)
                        return no_filho, caminho, nos_alcancados_qtd + 1, nos_expandidos
                    nos_alcancados.add(coord)
                    nos_alcancados_qtd += 1
                    fronteira.adicionar(no_filho)

        return None, None, nos_alcancados_qtd, nos_expandidos


class Dfs:
    def dfs(matriz, estado_inicial):
        nos_alcancados = set()
        nos_expandidos = 0
        pais = {}

        estado_inicial = No(
            estado_inicial[0],
            estado_inicial[1],
            matriz[estado_inicial[0]][estado_inicial[1]],
        )
        coord_inicial = (estado_inicial.i, estado_inicial.j)
        pais[coord_inicial] = None

        if ambiente.estado_final(estado_inicial):
            return estado_inicial, [coord_inicial], 1, 0

        fronteira = FronteiraDFS()
        fronteira.adicionar(estado_inicial)
        nos_alcancados.add(coord_inicial)
        nos_alcancados_qtd = 1

        while not fronteira.esta_vazia():
            no = fronteira.remover()
            nos_expandidos += 1

            for no_filho in ambiente.funcao_transicao(no):
                coord = (no_filho.i, no_filho.j)
                if coord not in nos_alcancados:
                    pais[coord] = (no.i, no.j)
                    if ambiente.estado_final(no_filho):
                        caminho = reconstruir_caminho(pais, coord)
                        return no_filho, caminho, nos_alcancados_qtd + 1, nos_expandidos
                    nos_alcancados.add(coord)
                    nos_alcancados_qtd += 1
                    fronteira.adicionar(no_filho)

        return None, None, nos_alcancados_qtd, nos_expandidos


class AStar:
    def _objetivos_borda():
        objetivos = []
        for i in range(ambiente.altura):
            for j in range(ambiente.largura):
                if i != 0 and i != ambiente.altura - 1 and j != 0 and j != ambiente.largura - 1:
                    continue
                rotulo = ambiente.matriz[i][j]
                if rotulo in (" ", "A", "J", "M"):
                    objetivos.append((i, j))
        return objetivos

    def _heuristica(coord, objetivos):
        i, j = coord
        if not objetivos:
            return float("inf")
        return min(math.hypot(i - gi, j - gj) for gi, gj in objetivos)

    def astar(matriz, estado_inicial):
        nos_alcancados = set()
        nos_expandidos = 0
        pais = {}

        objetivos = AStar._objetivos_borda()
        if not objetivos:
            return None, None, 0, 0

        no_inicial = No(
            estado_inicial[0],
            estado_inicial[1],
            matriz[estado_inicial[0]][estado_inicial[1]],
        )
        coord_inicial = (no_inicial.i, no_inicial.j)
        pais[coord_inicial] = None

        if ambiente.estado_final(no_inicial):
            return no_inicial, [coord_inicial], 1, 0

        fronteira = FronteiraAStar()
        g_score = {coord_inicial: 0}

        nos_alcancados.add(coord_inicial)
        nos_alcancados_qtd = 1

        h0 = AStar._heuristica(coord_inicial, objetivos)
        fronteira.adicionar(h0, h0, coord_inicial, no_inicial)

        while not fronteira.esta_vazia():
            f, h, _, coord_atual, no_atual = fronteira.remover()
            g_atual = g_score.get(coord_atual)
            if g_atual is None:
                continue
            if f != g_atual + h:
                continue

            nos_expandidos += 1
            if ambiente.estado_final(no_atual):
                caminho = reconstruir_caminho(pais, coord_atual)
                return no_atual, caminho, nos_alcancados_qtd, nos_expandidos

            for no_filho in ambiente.funcao_transicao(no_atual):
                coord = (no_filho.i, no_filho.j)
                tent_g = g_atual + 1
                g_ant = g_score.get(coord)
                if g_ant is None or tent_g < g_ant:
                    g_score[coord] = tent_g
                    pais[coord] = coord_atual
                    if coord not in nos_alcancados:
                        nos_alcancados.add(coord)
                        nos_alcancados_qtd += 1
                    h_filho = AStar._heuristica(coord, objetivos)
                    fronteira.adicionar(tent_g + h_filho, h_filho, coord, no_filho)

        return None, None, nos_alcancados_qtd, nos_expandidos

Na parte da execução, testo os 4 arquivos disponibilizados na pasta de estados.

Carrego os arquivos a partir dos métodos de Ambiente, executo os 3 algoritmos e exibo as informações retornadas por cada método (estado objetivo, caminho, nós alcançados e nós expandidos).

Quando os algoritmos retornam None como nó objetivo,  invoco o método de calcular a pontução de Ambiente.

In [19]:
# Execução dos algoritmos no arquivo "entice.txt", que é um estado não resolvido, para testar o cálculo de caminho mínimo
arquivo = "estados/entice.txt"
ambiente = carregar_ambiente_txt(arquivo)
matriz = ambiente.matriz
estado_inicial = ambiente.posicao_cavalo

print("\nArquivo:", arquivo)
print("Inicial:", estado_inicial)

res_bfs, caminho_bfs, alc_bfs, exp_bfs = Bfs.bfs(matriz, estado_inicial)
print("Objetivo obito pela BFS:", None if res_bfs is None else (res_bfs.i, res_bfs.j))
print("Tamanho do caminho obtido pela BFS ", None if caminho_bfs is None else len(caminho_bfs) - 1)
if caminho_bfs is not None:
    print("Caminho BFS:", caminho_bfs)
print("Nós alcançados pela BFS:", alc_bfs) 
print("Nós expandidos pela BFS:", exp_bfs)

if res_bfs is None:
    print("Sem fuga (BFS). Pontuação do cavalo:", ambiente.calcular_pontuacao())

res_dfs, caminho_dfs, alc_dfs, exp_dfs = Dfs.dfs(matriz, estado_inicial)
print("Objetivo obito pela DFS:", None if res_dfs is None else (res_dfs.i, res_dfs.j))
print("Tamanho do caminho obtido pela DFS:", None if caminho_dfs is None else len(caminho_dfs) - 1)
if caminho_dfs is not None:
    print("Caminho DFS:", caminho_dfs)
print("Nós alcançados pela DFS:", alc_dfs)
print("Nós expandidos pela DFS:", exp_dfs)

if res_dfs is None:
    print("Sem fuga (DFS). Pontuação do cavalo:", ambiente.calcular_pontuacao())

res_astar, caminho_astar, alc_astar, exp_astar = AStar.astar(matriz, estado_inicial)
print("Objetivo obtido pelo A*:", None if res_astar is None else (res_astar.i, res_astar.j))
print("Tamanho do caminho obtido pelo A*:", None if caminho_astar is None else len(caminho_astar) - 1)
if caminho_astar is not None:
    print("Caminho A*:", caminho_astar)
print("Nós alcançados pelo A*:", alc_astar)
print("Nós expandidos pelo A*:", exp_astar)

if res_astar is None:
    print("Sem fuga (A*). Pontuação do cavalo:", ambiente.calcular_pontuacao())


Arquivo: estados/entice.txt
Inicial: (7, 7)
Objetivo obito pela BFS: (14, 8)
Tamanho do caminho obtido pela BFS  8
Caminho BFS: [(7, 7), (8, 7), (8, 8), (9, 8), (10, 8), (11, 8), (12, 8), (13, 8), (14, 8)]
Nós alcançados pela BFS: 76
Nós expandidos pela BFS: 59
Objetivo obito pela DFS: (6, 0)
Tamanho do caminho obtido pela DFS: 8
Caminho DFS: [(7, 7), (7, 6), (6, 6), (6, 5), (6, 4), (6, 3), (6, 2), (6, 1), (6, 0)]
Nós alcançados pela DFS: 17
Nós expandidos pela DFS: 8
Objetivo obtido pelo A*: (14, 8)
Tamanho do caminho obtido pelo A*: 8
Caminho A*: [(7, 7), (8, 7), (8, 8), (9, 8), (10, 8), (11, 8), (12, 8), (13, 8), (14, 8)]
Nós alcançados pelo A*: 20
Nós expandidos pelo A*: 12


In [20]:
# Execução dos algoritmos no arquivo "geometry.txt", que é um estado não resolvido, para testar o cálculo de caminho mínimo
arquivo = "estados/geometry.txt"
ambiente = carregar_ambiente_txt(arquivo)
matriz = ambiente.matriz
estado_inicial = ambiente.posicao_cavalo

print("\nArquivo:", arquivo)
print("Inicial:", estado_inicial)

res_bfs, caminho_bfs, alc_bfs, exp_bfs = Bfs.bfs(matriz, estado_inicial)
print("Objetivo obito pela BFS:", None if res_bfs is None else (res_bfs.i, res_bfs.j))
print("Tamanho do caminho obtido pela BFS ", None if caminho_bfs is None else len(caminho_bfs) - 1)
if caminho_bfs is not None:
    print("Caminho BFS:", caminho_bfs)
print("Nós alcançados pela BFS:", alc_bfs) 
print("Nós expandidos pela BFS:", exp_bfs)

if res_bfs is None:
    print("Sem fuga (BFS). Pontuação do cavalo:", ambiente.calcular_pontuacao())

res_dfs, caminho_dfs, alc_dfs, exp_dfs = Dfs.dfs(matriz, estado_inicial)
print("Objetivo obito pela DFS:", None if res_dfs is None else (res_dfs.i, res_dfs.j))
print("Tamanho do caminho obtido pela DFS:", None if caminho_dfs is None else len(caminho_dfs) - 1)
if caminho_dfs is not None:
    print("Caminho DFS:", caminho_dfs)
print("Nós alcançados pela DFS:", alc_dfs)
print("Nós expandidos pela DFS:", exp_dfs)

if res_dfs is None:
    print("Sem fuga (DFS). Pontuação do cavalo:", ambiente.calcular_pontuacao())

res_astar, caminho_astar, alc_astar, exp_astar = AStar.astar(matriz, estado_inicial)
print("Objetivo obtido pelo A*:", None if res_astar is None else (res_astar.i, res_astar.j))
print("Tamanho do caminho obtido pelo A*:", None if caminho_astar is None else len(caminho_astar) - 1)
if caminho_astar is not None:
    print("Caminho A*:", caminho_astar)
print("Nós alcançados pelo A*:", alc_astar)
print("Nós expandidos pelo A*:", exp_astar)

if res_astar is None:
    print("Sem fuga (A*). Pontuação do cavalo:", ambiente.calcular_pontuacao())


Arquivo: estados/geometry.txt
Inicial: (15, 15)
Objetivo obito pela BFS: (29, 21)
Tamanho do caminho obtido pela BFS  20
Caminho BFS: [(15, 15), (16, 15), (17, 15), (18, 15), (18, 16), (18, 17), (18, 18), (19, 18), (20, 18), (21, 18), (22, 18), (23, 18), (24, 18), (25, 18), (26, 18), (26, 19), (26, 20), (27, 20), (28, 20), (28, 21), (29, 21)]
Nós alcançados pela BFS: 308
Nós expandidos pela BFS: 278
Objetivo obito pela DFS: (0, 18)
Tamanho do caminho obtido pela DFS: 24
Caminho DFS: [(15, 15), (15, 16), (15, 17), (15, 18), (15, 19), (14, 19), (13, 19), (13, 18), (12, 18), (11, 18), (11, 19), (10, 19), (9, 19), (9, 18), (8, 18), (7, 18), (7, 19), (6, 19), (5, 19), (4, 19), (4, 18), (3, 18), (2, 18), (1, 18), (0, 18)]
Nós alcançados pela DFS: 70
Nós expandidos pela DFS: 58
Objetivo obtido pelo A*: (29, 21)
Tamanho do caminho obtido pelo A*: 20
Caminho A*: [(15, 15), (16, 15), (17, 15), (18, 15), (18, 16), (18, 17), (18, 18), (19, 18), (20, 18), (21, 18), (22, 18), (23, 18), (24, 18), (2

In [21]:
# Execução dos algoritmos no arquivo "entice-resolvido.txt", que é um estado resolvido, para testar o cálculo de pontuação do cavalo quando não há fuga
arquivo = "estados/entice-resolvido.txt"
ambiente = carregar_ambiente_txt(arquivo)
matriz = ambiente.matriz
estado_inicial = ambiente.posicao_cavalo

print("\nArquivo:", arquivo)
print("Inicial:", estado_inicial)

res_bfs, caminho_bfs, alc_bfs, exp_bfs = Bfs.bfs(matriz, estado_inicial)
print("Objetivo obito pela BFS:", None if res_bfs is None else (res_bfs.i, res_bfs.j))
print("Tamanho do caminho obtido pela BFS ", None if caminho_bfs is None else len(caminho_bfs) - 1)
if caminho_bfs is not None:
    print("Caminho BFS:", caminho_bfs)
print("Nós alcançados pela BFS:", alc_bfs) 
print("Nós expandidos pela BFS:", exp_bfs)

if res_bfs is None:
    print("Sem fuga (BFS). Pontuação do cavalo:", ambiente.calcular_pontuacao())

res_dfs, caminho_dfs, alc_dfs, exp_dfs = Dfs.dfs(matriz, estado_inicial)
print("Objetivo obito pela DFS:", None if res_dfs is None else (res_dfs.i, res_dfs.j))
print("Tamanho do caminho obtido pela DFS ", None if caminho_dfs is None else len(caminho_dfs) - 1)
if caminho_dfs is not None:
    print("Caminho DFS:", caminho_dfs)
print("Nós alcançados pela DFS:", alc_dfs)
print("Nós expandidos pela DFS:", exp_dfs)

if res_dfs is None:
    print("Sem fuga (DFS). Pontuação do cavalo:", ambiente.calcular_pontuacao())

res_astar, caminho_astar, alc_astar, exp_astar = AStar.astar(matriz, estado_inicial)
print("Objetivo obtido pelo A*:", None if res_astar is None else (res_astar.i, res_astar.j))
print("Tamanho do caminho obtido pelo A*:", None if caminho_astar is None else len(caminho_astar) - 1)
if caminho_astar is not None:
    print("Caminho A*:", caminho_astar)
print("Nós alcançados pelo A*:", alc_astar)
print("Nós expandidos pelo A*:", exp_astar)

if res_astar is None:
    print("Sem fuga (A*). Pontuação do cavalo:", ambiente.calcular_pontuacao())


Arquivo: estados/entice-resolvido.txt
Inicial: (7, 7)
Objetivo obito pela BFS: None
Tamanho do caminho obtido pela BFS  None
Nós alcançados pela BFS: 56
Nós expandidos pela BFS: 56
Sem fuga (BFS). Pontuação do cavalo: 59
Objetivo obito pela DFS: None
Tamanho do caminho obtido pela DFS  None
Nós alcançados pela DFS: 56
Nós expandidos pela DFS: 56
Sem fuga (DFS). Pontuação do cavalo: 59
Objetivo obtido pelo A*: None
Tamanho do caminho obtido pelo A*: None
Nós alcançados pelo A*: 56
Nós expandidos pelo A*: 56
Sem fuga (A*). Pontuação do cavalo: 59


In [22]:
# Execução dos algoritmos no arquivo "geometry-resolvido.txt", que é um estado resolvido, para testar o cálculo de pontuação do cavalo quando não há fuga
arquivo = "estados/geometry-resolvido.txt"
ambiente = carregar_ambiente_txt(arquivo)
matriz = ambiente.matriz
estado_inicial = ambiente.posicao_cavalo

print("\nArquivo:", arquivo)
print("Inicial:", estado_inicial)

res_bfs, caminho_bfs, alc_bfs, exp_bfs = Bfs.bfs(matriz, estado_inicial)
print("Objetivo obito pela BFS:", None if res_bfs is None else (res_bfs.i, res_bfs.j))
print("Tamanho do caminho obtido pela BFS ", None if caminho_bfs is None else len(caminho_bfs) - 1)
if caminho_bfs is not None:
    print("Caminho BFS:", caminho_bfs)
print("Nós alcançados pela BFS:", alc_bfs) 
print("Nós expandidos pela BFS:", exp_bfs)

if res_bfs is None:
    print("Sem fuga (BFS). Pontuação do cavalo:", ambiente.calcular_pontuacao())

res_dfs, caminho_dfs, alc_dfs, exp_dfs = Dfs.dfs(matriz, estado_inicial)
print("Objetivo obito pela DFS:", None if res_dfs is None else (res_dfs.i, res_dfs.j))
print("Tamanho do caminho obtido pela DFS ", None if caminho_dfs is None else len(caminho_dfs) - 1)
if caminho_dfs is not None:
    print("Caminho DFS:", caminho_dfs)
print("Nós alcançados pela DFS:", alc_dfs)
print("Nós expandidos pela DFS:", exp_dfs)

if res_dfs is None:
    print("Sem fuga (DFS). Pontuação do cavalo:", ambiente.calcular_pontuacao())

res_astar, caminho_astar, alc_astar, exp_astar = AStar.astar(matriz, estado_inicial)
print("Objetivo obtido pelo A*:", None if res_astar is None else (res_astar.i, res_astar.j))
print("Tamanho do caminho obtido pelo A*:", None if caminho_astar is None else len(caminho_astar) - 1)
if caminho_astar is not None:
    print("Caminho A*:", caminho_astar)
print("Nós alcançados pelo A*:", alc_astar)
print("Nós expandidos pelo A*:", exp_astar)

if res_astar is None:
    print("Sem fuga (A*). Pontuação do cavalo:", ambiente.calcular_pontuacao())


Arquivo: estados/geometry-resolvido.txt
Inicial: (15, 15)
Objetivo obito pela BFS: None
Tamanho do caminho obtido pela BFS  None
Nós alcançados pela BFS: 322
Nós expandidos pela BFS: 322
Sem fuga (BFS). Pontuação do cavalo: 322
Objetivo obito pela DFS: None
Tamanho do caminho obtido pela DFS  None
Nós alcançados pela DFS: 322
Nós expandidos pela DFS: 322
Sem fuga (DFS). Pontuação do cavalo: 322
Objetivo obtido pelo A*: None
Tamanho do caminho obtido pelo A*: None
Nós alcançados pelo A*: 322
Nós expandidos pelo A*: 322
Sem fuga (A*). Pontuação do cavalo: 322


## 6) Discussão
A partir dos resultados obtidos, podemos observar a não otimalidade do DFS. Conforme estudamos em sala, esse algoritmo retorna o primeiro estado objetivo que encontra, e o caminho percorrido até esse estado objetivo não necessariamente é mínimo. Conforme comentado na segunda seção, o resultado da execução desse algoritmo pode mudar bastante dependendo da ordem que a função de transição considera as ações, pois isso altera a ordem em que os nós entram e saem da fronteira. Nessa instância e com essa ordem de ações, o DFS alcançou e expandiu menos nós, mas isso nem sempre é garantido.

Por outro lado, podemos constatar a otimalidade dos algoritmos BFS e A*, que sempre encontraram o caminho mínimo. Nessa instância e com a heurística de distância euclidiana que escolhi, o A* alcançou e expandiu menos nós, mas essa métricas poderiam ser maiores (ou menores) dependendo da heurística. O BFS alcançou e expandiu mais nós que o A*, mas não precisei me preocupar em encontrar uma heurística admissível para o problema.

Sobre os fatores de complexidade do problema, o fator de ramificação (b) é 4, já que em cada estado existem no máximo quatro ações possíveis (baixo, cima, direita e esquerda). Esse b é especialmente crítico no BFS, pois ele expande por níveis e a quantidade de estados cresce aproximadamente b<sup>d</sup>, onde d é a profundidade mínima da solução ótima (número de ações do menor caminho até uma saída). Isso ajuda a explicar por que, no geometry (d maior), a BFS expandiu 278 nós, enquanto no entice expandiu 59. Já no DFS, o parâmetro mais sensível é o tamanho do caminho mais longo possível (m). O DFS pode seguir um ramo profundo por muito tempo antes de voltar, então o custo pode piorar bastante quando m é grande. No A*, esses mesmos fatores também influenciam, mas a heurística admissível faz com que não expandamos às cegas e guia a busca para nós com menor f(n). Por isso, nos casos com fuga, ele expandiu bem menos nós que o BFS (12 vs 59 no entice; 123 vs 278 no geometry).



